# Landslide combined-class step 09: parish-level avoided EAD summaries

This notebook allocates combined-class landslide avoided EADs to Jamaica parishes.

Method:
- reads the minimum and maximum combined-class asset map layers from step 06;
- spatially joins point assets to `admin1` parish polygons;
- allocates line assets by length share where they cross parish boundaries;
- allocates polygon assets by area share where they cross parish boundaries;
- allocates any residual geometry outside parish polygons to the nearest parish so national avoided-EAD totals are preserved;
- reports annual EAD and 50-year present-value EAD using the same 10% discount rate as the coastal and river flood analyses.

Important: the step-06 combined-class geospatial map layers include assets with non-zero forest-scenario change. They preserve avoided-damage totals, but they are not full national exposure layers. Parish percentages in this notebook are therefore shares of national avoided or increased damages, not parish-level percentages of all baseline EAD.

In [ ]:
from pathlib import Path

from IPython.display import display
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.max_columns = 120
pd.options.display.float_format = '{:,.3f}'.format

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
landslide_root = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages'
output_dir = landslide_root / 'parish_analysis_source_and_runout_zones_combined_class'
output_dir.mkdir(parents=True, exist_ok=True)

admin_boundaries_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg'

scenario_paths = {
    'minimum': landslide_root / 'results_landslide_minimum_scenario_combined_class/damage_estimates/landslide_source_and_runout_ead_asset_map_layers_combined_class.gpkg',
    'maximum': landslide_root / 'results_landslide_maximum_scenario_combined_class/damage_estimates/landslide_source_and_runout_ead_asset_map_layers_combined_class.gpkg',
}

for required_path in [admin_boundaries_path, *scenario_paths.values()]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print('Output directory:', output_dir)
print('Admin boundaries:', admin_boundaries_path)
for scenario_name, scenario_path in scenario_paths.items():
    print(f'{scenario_name}:', scenario_path)

## Discount factor

The discount factor matches the coastal-flood long-timeframe calculation: 50 years, 10% discount rate, including year 0.

In [ ]:
discount_years = 50
discount_rate = 0.10
discounted_suffix = 'PV_50Y_10pct'


def compute_discount_factor(years: int, annual_discount_rate: float) -> float:
    return sum(1 / (1 + annual_discount_rate) ** year for year in range(years + 1))


discount_factor = compute_discount_factor(discount_years, discount_rate)

discount_metadata = pd.DataFrame([
    {
        'Discount_Years': discount_years,
        'Discount_Rate': discount_rate,
        'Discount_Factor': discount_factor,
        'Method': 'sum(1 / (1 + discount_rate) ** year for year in range(years + 1))',
        'Includes_Year_0': True,
    }
])

discount_metadata_file = output_dir / 'landslide_parish_discount_factor_10pct_combined_class.csv'
discount_metadata.to_csv(discount_metadata_file, index=False)
print('Saved:', discount_metadata_file)
display(discount_metadata)

## Load parish boundaries

In [ ]:
parishes = gpd.read_file(admin_boundaries_path, layer='admin1')
if parishes.crs is None:
    raise ValueError('Parish boundaries have no CRS.')
if str(parishes.crs).upper() != 'EPSG:3448':
    parishes = parishes.to_crs('EPSG:3448')

parishes = parishes[['PARISH', 'CODE', 'geometry']].copy()
parishes = parishes.rename(columns={'PARISH': 'ParishName', 'CODE': 'ParishCode'})
parishes['ParishName'] = parishes['ParishName'].astype(str).str.strip()
parishes = parishes[parishes.geometry.notna()].copy()
parishes = parishes[~parishes.geometry.is_empty].copy()
parishes['geometry'] = parishes.geometry.make_valid()

print(f'Loaded {len(parishes)} parishes')
display(parishes[['ParishName', 'ParishCode']].sort_values('ParishName'))

## Helper functions

In [ ]:
asset_identity_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
source_metric_cols = [
    'EAD_Baseline_USD',
    'EAD_Deforestation_USD',
    'EAD_Reafforestation_USD',
    'Avoided_EAD_Protection_USD',
    'Avoided_EAD_Reafforestation_USD',
    'Combined_Benefit_Reafforestation_vs_Deforestation_USD',
]
benefit_metric_cols = [
    'Protection_Net_Avoided_EAD_USD',
    'Protection_Positive_Avoided_EAD_USD',
    'Protection_Increased_Damage_USD',
    'Reafforestation_Net_Avoided_EAD_USD',
    'Reafforestation_Positive_Avoided_EAD_USD',
    'Reafforestation_Increased_Damage_USD',
    'Combined_Benefit_Reafforestation_vs_Deforestation_USD',
]
allocation_metric_cols = source_metric_cols + [
    'Protection_Net_Avoided_EAD_USD',
    'Protection_Positive_Avoided_EAD_USD',
    'Protection_Increased_Damage_USD',
    'Reafforestation_Net_Avoided_EAD_USD',
    'Reafforestation_Positive_Avoided_EAD_USD',
    'Reafforestation_Increased_Damage_USD',
]
allocation_metric_cols = list(dict.fromkeys(allocation_metric_cols))


def format_usd_readable(value):
    if pd.isna(value):
        return 'NA'
    value = float(value)
    sign = '-' if value < 0 else ''
    abs_value = abs(value)
    if abs_value >= 1_000_000_000:
        return f'{sign}US${abs_value / 1_000_000_000:,.2f} billion'
    if abs_value >= 1_000_000:
        return f'{sign}US${abs_value / 1_000_000:,.2f} million'
    if abs_value >= 1_000:
        return f'{sign}US${abs_value / 1_000:,.1f} thousand'
    return f'{sign}US${abs_value:,.0f}'


def format_pct(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.1f}%'


def add_benefit_metrics(gdf):
    out = gdf.copy()
    out['Protection_Net_Avoided_EAD_USD'] = pd.to_numeric(out['Avoided_EAD_Protection_USD'], errors='coerce').fillna(0.0)
    out['Reafforestation_Net_Avoided_EAD_USD'] = pd.to_numeric(out['Avoided_EAD_Reafforestation_USD'], errors='coerce').fillna(0.0)
    out['Protection_Positive_Avoided_EAD_USD'] = out['Protection_Net_Avoided_EAD_USD'].clip(lower=0.0)
    out['Reafforestation_Positive_Avoided_EAD_USD'] = out['Reafforestation_Net_Avoided_EAD_USD'].clip(lower=0.0)
    out['Protection_Increased_Damage_USD'] = (-out['Protection_Net_Avoided_EAD_USD']).clip(lower=0.0)
    out['Reafforestation_Increased_Damage_USD'] = (-out['Reafforestation_Net_Avoided_EAD_USD']).clip(lower=0.0)
    return out


def nearest_parish_names(geometries, parish_gdf):
    names = []
    parish_geometries = parish_gdf.geometry
    parish_names = parish_gdf['ParishName'].to_numpy()
    for geom in geometries:
        if geom is None or geom.is_empty:
            names.append('Unassigned')
            continue
        point = geom.representative_point() if geom.geom_type not in {'Point', 'MultiPoint'} else geom.centroid
        distances = parish_geometries.distance(point)
        names.append(str(parish_names[int(distances.idxmin())]))
    return names


def multiply_metrics_by_weight(frame, metric_cols, weight_col='Allocation_Weight'):
    out = frame.copy()
    for metric_col in metric_cols:
        out[metric_col] = pd.to_numeric(out[metric_col], errors='coerce').fillna(0.0) * out[weight_col]
    return out


def allocate_points_to_parishes(points, parish_gdf, metric_cols):
    if points.empty:
        return gpd.GeoDataFrame(columns=[*points.columns, 'ParishName', 'Allocation_Weight', 'Allocation_Method'], geometry='geometry', crs='EPSG:3448')

    joined = gpd.sjoin(
        points,
        parish_gdf[['ParishName', 'geometry']],
        how='left',
        predicate='within',
    ).drop(columns=['index_right'], errors='ignore')

    missing_mask = joined['ParishName'].isna()
    if missing_mask.any():
        joined.loc[missing_mask, 'ParishName'] = nearest_parish_names(joined.loc[missing_mask].geometry, parish_gdf)
        joined.loc[missing_mask, 'Allocation_Method'] = 'nearest_point_fallback'

    joined['Allocation_Method'] = joined.get('Allocation_Method', pd.Series(index=joined.index, dtype='object')).fillna('point_within')
    joined['Allocation_Weight'] = 1.0
    return multiply_metrics_by_weight(joined, metric_cols)


def allocate_measured_geometries_to_parishes(features, parish_gdf, metric_cols, measure_kind):
    if features.empty:
        return gpd.GeoDataFrame(columns=[*features.columns, 'ParishName', 'Allocation_Weight', 'Allocation_Method'], geometry='geometry', crs='EPSG:3448')

    features = features.copy()
    if measure_kind == 'length':
        features['Original_Measure'] = features.geometry.length
        method = 'length_share'
    elif measure_kind == 'area':
        features['Original_Measure'] = features.geometry.area
        method = 'area_share'
    else:
        raise ValueError(measure_kind)

    positive_measure = features['Original_Measure'] > 0
    zero_measure = features.loc[~positive_measure].copy()
    features = features.loc[positive_measure].copy()

    pieces = []
    if not features.empty:
        overlay = gpd.overlay(
            features,
            parish_gdf[['ParishName', 'geometry']],
            how='intersection',
            keep_geom_type=False,
        )
        if not overlay.empty:
            overlay = overlay[overlay.geometry.notna()].copy()
            overlay = overlay[~overlay.geometry.is_empty].copy()
            if measure_kind == 'length':
                overlay['Piece_Measure'] = overlay.geometry.length
            else:
                overlay['Piece_Measure'] = overlay.geometry.area
            overlay = overlay[overlay['Piece_Measure'] > 0].copy()
            overlay['Allocation_Weight_Raw'] = overlay['Piece_Measure'] / overlay['Original_Measure']
            raw_weight_sum = overlay.groupby('Feature_ID')['Allocation_Weight_Raw'].transform('sum')
            overlay['Allocation_Weight'] = np.where(
                raw_weight_sum > 1.0,
                overlay['Allocation_Weight_Raw'] / raw_weight_sum,
                overlay['Allocation_Weight_Raw'],
            )
            overlay['Allocation_Method'] = np.where(raw_weight_sum > 1.000001, f'{method}_normalised', method)
            pieces.append(multiply_metrics_by_weight(overlay, metric_cols))

        if pieces:
            allocated_weights = pieces[0].groupby('Feature_ID')['Allocation_Weight'].sum()
        else:
            allocated_weights = pd.Series(dtype='float64')

        residual = features.copy()
        residual['Allocated_Weight'] = residual['Feature_ID'].map(allocated_weights).fillna(0.0)
        residual['Allocation_Weight'] = (1.0 - residual['Allocated_Weight']).clip(lower=0.0)
        residual = residual[residual['Allocation_Weight'] > 1e-6].copy()
        if not residual.empty:
            residual['ParishName'] = nearest_parish_names(residual.geometry, parish_gdf)
            residual['Allocation_Method'] = f'nearest_residual_{measure_kind}'
            pieces.append(multiply_metrics_by_weight(residual, metric_cols))

    if not zero_measure.empty:
        zero_measure['ParishName'] = nearest_parish_names(zero_measure.geometry, parish_gdf)
        zero_measure['Allocation_Method'] = f'nearest_zero_{measure_kind}'
        zero_measure['Allocation_Weight'] = 1.0
        pieces.append(multiply_metrics_by_weight(zero_measure, metric_cols))

    if not pieces:
        return gpd.GeoDataFrame(columns=[*features.columns, 'ParishName', 'Allocation_Weight', 'Allocation_Method'], geometry='geometry', crs='EPSG:3448')

    return gpd.GeoDataFrame(pd.concat(pieces, ignore_index=True), geometry='geometry', crs='EPSG:3448')


def allocate_scenario_to_parishes(scenario_name, map_layer_path):
    gdf = gpd.read_file(map_layer_path)
    if gdf.crs is None:
        raise ValueError(f'{map_layer_path} has no CRS')
    if str(gdf.crs).upper() != 'EPSG:3448':
        gdf = gdf.to_crs('EPSG:3448')

    required_cols = asset_identity_cols + source_metric_cols + ['geometry']
    missing_cols = [col for col in required_cols if col not in gdf.columns]
    if missing_cols:
        raise KeyError(f'{scenario_name} missing columns: {missing_cols}')

    gdf = gdf[required_cols].copy()
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    gdf['geometry'] = gdf.geometry.make_valid()
    gdf = add_benefit_metrics(gdf)
    gdf['Scenario'] = scenario_name
    gdf['Feature_ID'] = [f'{scenario_name}_{idx}' for idx in range(len(gdf))]

    geom_type = gdf.geometry.geom_type.astype(str)
    points = gdf[geom_type.str.contains('Point', na=False)].copy()
    lines = gdf[geom_type.str.contains('LineString', na=False)].copy()
    polygons = gdf[geom_type.str.contains('Polygon', na=False)].copy()
    other = gdf[~(geom_type.str.contains('Point|LineString|Polygon', na=False))].copy()

    pieces = [
        allocate_points_to_parishes(points, parishes, allocation_metric_cols),
        allocate_measured_geometries_to_parishes(lines, parishes, allocation_metric_cols, 'length'),
        allocate_measured_geometries_to_parishes(polygons, parishes, allocation_metric_cols, 'area'),
    ]

    if not other.empty:
        other['ParishName'] = nearest_parish_names(other.geometry, parishes)
        other['Allocation_Method'] = 'nearest_other_geometry'
        other['Allocation_Weight'] = 1.0
        pieces.append(multiply_metrics_by_weight(other, allocation_metric_cols))

    allocated = gpd.GeoDataFrame(pd.concat([piece for piece in pieces if not piece.empty], ignore_index=True), geometry='geometry', crs='EPSG:3448')
    allocated['Scenario'] = scenario_name

    input_totals = gdf[allocation_metric_cols].sum(numeric_only=True)
    allocated_totals = allocated[allocation_metric_cols].sum(numeric_only=True)
    qc = pd.DataFrame({
        'Metric': allocation_metric_cols,
        'Input_Total_USD': [float(input_totals.get(col, 0.0)) for col in allocation_metric_cols],
        'Allocated_Total_USD': [float(allocated_totals.get(col, 0.0)) for col in allocation_metric_cols],
    })
    qc['Difference_USD'] = qc['Allocated_Total_USD'] - qc['Input_Total_USD']
    qc['Abs_Difference_USD'] = qc['Difference_USD'].abs()
    qc['Scenario'] = scenario_name

    method_summary = (
        allocated.groupby('Allocation_Method', as_index=False)
        .agg(
            Piece_Count=('Feature_ID', 'size'),
            Feature_Count=('Feature_ID', lambda values: values.astype(str).nunique()),
            Protection_Positive_Avoided_EAD_USD=('Protection_Positive_Avoided_EAD_USD', 'sum'),
            Reafforestation_Positive_Avoided_EAD_USD=('Reafforestation_Positive_Avoided_EAD_USD', 'sum'),
        )
        .sort_values('Piece_Count', ascending=False)
    )
    method_summary['Scenario'] = scenario_name

    return gdf, allocated, qc, method_summary

## Allocate asset changes to parishes

In [ ]:
scenario_inputs = {}
allocated_pieces = []
qc_tables = []
method_tables = []

for scenario_name, map_layer_path in scenario_paths.items():
    print(f'Allocating {scenario_name} scenario...')
    source_gdf, allocated_gdf, qc, method_summary = allocate_scenario_to_parishes(scenario_name, map_layer_path)
    scenario_inputs[scenario_name] = source_gdf
    allocated_pieces.append(allocated_gdf)
    qc_tables.append(qc)
    method_tables.append(method_summary)
    print(f'  source features: {len(source_gdf):,}')
    print(f'  allocated pieces: {len(allocated_gdf):,}')
    print(f'  max absolute QC difference: {qc["Abs_Difference_USD"].max():,.6f}')

allocated_all = gpd.GeoDataFrame(pd.concat(allocated_pieces, ignore_index=True), geometry='geometry', crs='EPSG:3448')
allocation_qc = pd.concat(qc_tables, ignore_index=True)
allocation_methods = pd.concat(method_tables, ignore_index=True)

allocation_qc_file = output_dir / 'landslide_parish_allocation_qc_source_and_runout_zones_combined_class.csv'
allocation_methods_file = output_dir / 'landslide_parish_allocation_methods_source_and_runout_zones_combined_class.csv'
allocated_gpkg_file = output_dir / 'landslide_parish_allocated_avoided_ead_pieces_source_and_runout_zones_combined_class.gpkg'

allocation_qc.to_csv(allocation_qc_file, index=False)
allocation_methods.to_csv(allocation_methods_file, index=False)
allocated_all.to_file(allocated_gpkg_file, driver='GPKG')

print('Saved:', allocation_qc_file)
print('Saved:', allocation_methods_file)
print('Saved:', allocated_gpkg_file)
display(allocation_qc.sort_values(['Scenario', 'Abs_Difference_USD'], ascending=[True, False]).head(20))
display(allocation_methods.sort_values(['Scenario', 'Piece_Count'], ascending=[True, False]))

## Parish summaries

In [ ]:
def add_discounted_and_readable_columns(summary):
    out = summary.copy()
    for metric_col in benefit_metric_cols:
        if metric_col in out.columns:
            out[f'{metric_col}_{discounted_suffix}'] = out[metric_col] * discount_factor

    readable_cols = [
        'Protection_Net_Avoided_EAD_USD',
        'Protection_Positive_Avoided_EAD_USD',
        'Protection_Increased_Damage_USD',
        'Reafforestation_Net_Avoided_EAD_USD',
        'Reafforestation_Positive_Avoided_EAD_USD',
        'Reafforestation_Increased_Damage_USD',
        f'Protection_Positive_Avoided_EAD_USD_{discounted_suffix}',
        f'Protection_Increased_Damage_USD_{discounted_suffix}',
        f'Reafforestation_Positive_Avoided_EAD_USD_{discounted_suffix}',
        f'Reafforestation_Increased_Damage_USD_{discounted_suffix}',
    ]
    for col in readable_cols:
        if col in out.columns:
            out[f'{col}_Readable'] = out[col].apply(format_usd_readable)

    pct_cols = [col for col in out.columns if col.startswith('Pct_of_National_')]
    for col in pct_cols:
        out[f'{col}_Label'] = out[col].apply(format_pct)
    return out


def add_national_shares(summary, group_cols):
    out = summary.copy()
    share_metrics = [
        'Protection_Net_Avoided_EAD_USD',
        'Protection_Positive_Avoided_EAD_USD',
        'Protection_Increased_Damage_USD',
        'Reafforestation_Net_Avoided_EAD_USD',
        'Reafforestation_Positive_Avoided_EAD_USD',
        'Reafforestation_Increased_Damage_USD',
        'Combined_Benefit_Reafforestation_vs_Deforestation_USD',
    ]
    for metric_col in share_metrics:
        if metric_col not in out.columns:
            continue
        denom = out.groupby('Scenario')[metric_col].transform('sum')
        out[f'Pct_of_National_{metric_col}'] = np.where(denom != 0, 100.0 * out[metric_col] / denom, 0.0)
    return out


summary_metrics = benefit_metric_cols.copy()
parish_summary = (
    allocated_all
    .groupby(['Scenario', 'ParishName'], as_index=False)
    .agg(
        **{metric_col: (metric_col, 'sum') for metric_col in summary_metrics},
        Contributing_Feature_Count=('Feature_ID', lambda values: values.astype(str).nunique()),
        Allocated_Piece_Count=('Feature_ID', 'size'),
    )
)
parish_summary = add_national_shares(parish_summary, ['Scenario', 'ParishName'])
parish_summary = add_discounted_and_readable_columns(parish_summary)

sector_parish_summary = (
    allocated_all
    .groupby(['Scenario', 'ParishName', 'Sector'], as_index=False)
    .agg(
        **{metric_col: (metric_col, 'sum') for metric_col in summary_metrics},
        Contributing_Feature_Count=('Feature_ID', lambda values: values.astype(str).nunique()),
        Allocated_Piece_Count=('Feature_ID', 'size'),
    )
)
sector_parish_summary = add_national_shares(sector_parish_summary, ['Scenario', 'ParishName', 'Sector'])
sector_parish_summary = add_discounted_and_readable_columns(sector_parish_summary)

subsector_parish_summary = (
    allocated_all
    .groupby(['Scenario', 'ParishName', 'Sector', 'Subsector'], as_index=False)
    .agg(
        **{metric_col: (metric_col, 'sum') for metric_col in summary_metrics},
        Contributing_Feature_Count=('Feature_ID', lambda values: values.astype(str).nunique()),
        Allocated_Piece_Count=('Feature_ID', 'size'),
    )
)
subsector_parish_summary = add_national_shares(subsector_parish_summary, ['Scenario', 'ParishName', 'Sector', 'Subsector'])
subsector_parish_summary = add_discounted_and_readable_columns(subsector_parish_summary)

for scenario_name in scenario_paths:
    scenario_file = output_dir / f'landslide_ead_parish_summary_{scenario_name}_source_and_runout_zones_combined_class.csv'
    parish_summary.loc[parish_summary['Scenario'] == scenario_name].sort_values(
        'Protection_Positive_Avoided_EAD_USD', ascending=False
    ).to_csv(scenario_file, index=False)
    print('Saved:', scenario_file)

parish_summary_file = output_dir / 'landslide_ead_parish_summary_all_scenarios_source_and_runout_zones_combined_class.csv'
sector_parish_summary_file = output_dir / 'landslide_ead_parish_sector_summary_all_scenarios_source_and_runout_zones_combined_class.csv'
subsector_parish_summary_file = output_dir / 'landslide_ead_parish_subsector_summary_all_scenarios_source_and_runout_zones_combined_class.csv'

parish_summary.to_csv(parish_summary_file, index=False)
sector_parish_summary.to_csv(sector_parish_summary_file, index=False)
subsector_parish_summary.to_csv(subsector_parish_summary_file, index=False)

print('Saved:', parish_summary_file)
print('Saved:', sector_parish_summary_file)
print('Saved:', subsector_parish_summary_file)

display(parish_summary.sort_values(['Scenario', 'Protection_Positive_Avoided_EAD_USD'], ascending=[True, False]).head(20))

## Minimum-maximum parish ranges

In [ ]:
range_metric_cols = [
    'Protection_Net_Avoided_EAD_USD',
    'Protection_Positive_Avoided_EAD_USD',
    'Protection_Increased_Damage_USD',
    'Reafforestation_Net_Avoided_EAD_USD',
    'Reafforestation_Positive_Avoided_EAD_USD',
    'Reafforestation_Increased_Damage_USD',
    'Combined_Benefit_Reafforestation_vs_Deforestation_USD',
    f'Protection_Positive_Avoided_EAD_USD_{discounted_suffix}',
    f'Protection_Increased_Damage_USD_{discounted_suffix}',
    f'Reafforestation_Positive_Avoided_EAD_USD_{discounted_suffix}',
    f'Reafforestation_Increased_Damage_USD_{discounted_suffix}',
    'Pct_of_National_Protection_Positive_Avoided_EAD_USD',
    'Pct_of_National_Protection_Increased_Damage_USD',
    'Pct_of_National_Reafforestation_Positive_Avoided_EAD_USD',
    'Pct_of_National_Reafforestation_Increased_Damage_USD',
]


def build_min_max_ranges(summary, group_cols):
    scenario_values = {}
    for scenario_name in ['minimum', 'maximum']:
        scenario_values[scenario_name] = (
            summary.loc[summary['Scenario'] == scenario_name, group_cols + range_metric_cols]
            .copy()
            .set_index(group_cols)
        )

    all_index = scenario_values['minimum'].index.union(scenario_values['maximum'].index)
    pieces = []
    for scenario_name, prefix in [('minimum', 'Min'), ('maximum', 'Max')]:
        scenario_df = scenario_values[scenario_name].reindex(all_index).fillna(0.0)
        scenario_df = scenario_df.rename(columns={col: f'{prefix}_{col}' for col in range_metric_cols})
        pieces.append(scenario_df)

    combined = pd.concat(pieces, axis=1).reset_index()

    for metric_col in range_metric_cols:
        min_col = f'Min_{metric_col}'
        max_col = f'Max_{metric_col}'
        range_min_col = f'{metric_col}_Range_Min'
        range_max_col = f'{metric_col}_Range_Max'
        combined[range_min_col] = combined[[min_col, max_col]].min(axis=1)
        combined[range_max_col] = combined[[min_col, max_col]].max(axis=1)

        if metric_col.endswith('_USD') or f'_{discounted_suffix}' in metric_col:
            combined[f'{range_min_col}_Readable'] = combined[range_min_col].apply(format_usd_readable)
            combined[f'{range_max_col}_Readable'] = combined[range_max_col].apply(format_usd_readable)
        if metric_col.startswith('Pct_of_National_'):
            combined[f'{range_min_col}_Label'] = combined[range_min_col].apply(format_pct)
            combined[f'{range_max_col}_Label'] = combined[range_max_col].apply(format_pct)

    return combined


parish_min_max = build_min_max_ranges(parish_summary, ['ParishName'])
sector_parish_min_max = build_min_max_ranges(sector_parish_summary, ['ParishName', 'Sector'])
subsector_parish_min_max = build_min_max_ranges(subsector_parish_summary, ['ParishName', 'Sector', 'Subsector'])

parish_min_max = parish_min_max.sort_values('Protection_Positive_Avoided_EAD_USD_Range_Max', ascending=False)
sector_parish_min_max = sector_parish_min_max.sort_values('Protection_Positive_Avoided_EAD_USD_Range_Max', ascending=False)
subsector_parish_min_max = subsector_parish_min_max.sort_values('Protection_Positive_Avoided_EAD_USD_Range_Max', ascending=False)

parish_min_max_file = output_dir / 'landslide_ead_parish_summary_min_max_source_and_runout_zones_combined_class.csv'
sector_parish_min_max_file = output_dir / 'landslide_ead_parish_sector_summary_min_max_source_and_runout_zones_combined_class.csv'
subsector_parish_min_max_file = output_dir / 'landslide_ead_parish_subsector_summary_min_max_source_and_runout_zones_combined_class.csv'

parish_min_max.to_csv(parish_min_max_file, index=False)
sector_parish_min_max.to_csv(sector_parish_min_max_file, index=False)
subsector_parish_min_max.to_csv(subsector_parish_min_max_file, index=False)

print('Saved:', parish_min_max_file)
print('Saved:', sector_parish_min_max_file)
print('Saved:', subsector_parish_min_max_file)

display(parish_min_max.head(14))

## Drafting table

In [ ]:
draft_cols = [
    'ParishName',
    'Protection_Positive_Avoided_EAD_USD_Range_Min_Readable',
    'Protection_Positive_Avoided_EAD_USD_Range_Max_Readable',
    'Pct_of_National_Protection_Positive_Avoided_EAD_USD_Range_Min_Label',
    'Pct_of_National_Protection_Positive_Avoided_EAD_USD_Range_Max_Label',
    'Protection_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Min_Readable',
    'Protection_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Max_Readable',
    'Protection_Increased_Damage_USD_Range_Min_Readable',
    'Protection_Increased_Damage_USD_Range_Max_Readable',
    'Reafforestation_Positive_Avoided_EAD_USD_Range_Min_Readable',
    'Reafforestation_Positive_Avoided_EAD_USD_Range_Max_Readable',
    'Pct_of_National_Reafforestation_Positive_Avoided_EAD_USD_Range_Min_Label',
    'Pct_of_National_Reafforestation_Positive_Avoided_EAD_USD_Range_Max_Label',
    'Reafforestation_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Min_Readable',
    'Reafforestation_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Max_Readable',
    'Reafforestation_Increased_Damage_USD_Range_Min_Readable',
    'Reafforestation_Increased_Damage_USD_Range_Max_Readable',
]

drafting_table = parish_min_max[draft_cols].copy()
drafting_table = drafting_table.rename(columns={
    'Protection_Positive_Avoided_EAD_USD_Range_Min_Readable': 'Protection_Avoided_EAD_Min',
    'Protection_Positive_Avoided_EAD_USD_Range_Max_Readable': 'Protection_Avoided_EAD_Max',
    'Pct_of_National_Protection_Positive_Avoided_EAD_USD_Range_Min_Label': 'Protection_Share_Min',
    'Pct_of_National_Protection_Positive_Avoided_EAD_USD_Range_Max_Label': 'Protection_Share_Max',
    'Protection_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Min_Readable': 'Protection_Avoided_EAD_50yr_PV_Min',
    'Protection_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Max_Readable': 'Protection_Avoided_EAD_50yr_PV_Max',
    'Protection_Increased_Damage_USD_Range_Min_Readable': 'Protection_Increased_Damage_Min',
    'Protection_Increased_Damage_USD_Range_Max_Readable': 'Protection_Increased_Damage_Max',
    'Reafforestation_Positive_Avoided_EAD_USD_Range_Min_Readable': 'Reafforestation_Avoided_EAD_Min',
    'Reafforestation_Positive_Avoided_EAD_USD_Range_Max_Readable': 'Reafforestation_Avoided_EAD_Max',
    'Pct_of_National_Reafforestation_Positive_Avoided_EAD_USD_Range_Min_Label': 'Reafforestation_Share_Min',
    'Pct_of_National_Reafforestation_Positive_Avoided_EAD_USD_Range_Max_Label': 'Reafforestation_Share_Max',
    'Reafforestation_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Min_Readable': 'Reafforestation_Avoided_EAD_50yr_PV_Min',
    'Reafforestation_Positive_Avoided_EAD_USD_PV_50Y_10pct_Range_Max_Readable': 'Reafforestation_Avoided_EAD_50yr_PV_Max',
    'Reafforestation_Increased_Damage_USD_Range_Min_Readable': 'Reafforestation_Increased_Damage_Min',
    'Reafforestation_Increased_Damage_USD_Range_Max_Readable': 'Reafforestation_Increased_Damage_Max',
})

drafting_table_file = output_dir / 'landslide_parish_drafting_values_readable_source_and_runout_zones_combined_class.csv'
drafting_table.to_csv(drafting_table_file, index=False)
print('Saved:', drafting_table_file)
display(drafting_table)

## Parish charts

In [ ]:
def plot_parish_range_bars(range_table, metric_col, title, output_png, top_count=14):
    min_col = f'{metric_col}_Range_Min'
    max_col = f'{metric_col}_Range_Max'
    plot_table = range_table[['ParishName', min_col, max_col]].copy()
    plot_table = plot_table.sort_values(max_col, ascending=True).tail(top_count)

    fig, ax = plt.subplots(figsize=(10, 7.2))
    y = np.arange(len(plot_table))
    min_values = plot_table[min_col].to_numpy() / 1_000_000
    max_values = plot_table[max_col].to_numpy() / 1_000_000

    ax.barh(y - 0.18, min_values, height=0.34, color='#9ecae1', edgecolor='#3a3a3a', linewidth=0.6, label='Minimum damage scenario')
    ax.barh(y + 0.18, max_values, height=0.34, color='#08519c', edgecolor='#3a3a3a', linewidth=0.6, label='Maximum damage scenario')
    ax.set_yticks(y)
    ax.set_yticklabels(plot_table['ParishName'])
    ax.set_xlabel('Avoided EAD (USD million per year)')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.25)
    ax.legend(frameon=True, loc='lower right')
    plt.tight_layout()
    fig.savefig(output_png, dpi=300, bbox_inches='tight')
    print('Saved:', output_png)
    plt.show()


plot_parish_range_bars(
    parish_min_max,
    'Protection_Positive_Avoided_EAD_USD',
    'Landslide avoided EAD from protecting existing forests by parish',
    output_dir / 'landslide_parish_protection_avoided_ead_min_max_source_and_runout_zones_combined_class.png',
)

plot_parish_range_bars(
    parish_min_max,
    'Reafforestation_Positive_Avoided_EAD_USD',
    'Landslide avoided EAD from reafforestation by parish',
    output_dir / 'landslide_parish_reafforestation_avoided_ead_min_max_source_and_runout_zones_combined_class.png',
)

## Quick interpretation values

In [ ]:
def top_parishes_for(metric_col, top_count=6):
    min_col = f'{metric_col}_Range_Min'
    max_col = f'{metric_col}_Range_Max'
    share_min_col = f'Pct_of_National_{metric_col}_Range_Min'
    share_max_col = f'Pct_of_National_{metric_col}_Range_Max'
    cols = ['ParishName', min_col, max_col, share_min_col, share_max_col]
    out = parish_min_max[cols].copy().sort_values(max_col, ascending=False).head(top_count)
    out[f'{metric_col}_Min_Readable'] = out[min_col].apply(format_usd_readable)
    out[f'{metric_col}_Max_Readable'] = out[max_col].apply(format_usd_readable)
    out[f'{metric_col}_Share_Min_Label'] = out[share_min_col].apply(format_pct)
    out[f'{metric_col}_Share_Max_Label'] = out[share_max_col].apply(format_pct)
    return out

print('Top parishes: protection avoided EAD')
display(top_parishes_for('Protection_Positive_Avoided_EAD_USD', 8))

print('Top parishes: reafforestation avoided EAD')
display(top_parishes_for('Reafforestation_Positive_Avoided_EAD_USD', 8))

print('National increased damage totals by scenario')
increase_totals = parish_summary.groupby('Scenario', as_index=False)[[
    'Protection_Increased_Damage_USD',
    'Reafforestation_Increased_Damage_USD',
]].sum()
increase_totals['Protection_Increased_Damage_Readable'] = increase_totals['Protection_Increased_Damage_USD'].apply(format_usd_readable)
increase_totals['Reafforestation_Increased_Damage_Readable'] = increase_totals['Reafforestation_Increased_Damage_USD'].apply(format_usd_readable)
display(increase_totals)